# Phase 2: Cleaning & ETL

In notebook 01 we measured the damage in the raw NYC taxi data: null/zero
passenger counts, negative fares, impossible trip distances, and pickup times
outside the labelled range or after the dropoff. Now we turn that into a real
cleaning step.

## Why a function, not inline filtering

We're writing this as a single function, `clean_trips(df) -> DataFrame`,
instead of a chain of `.filter()` calls scattered through the notebook.
Reasons:

- **Testable.** You can call `clean_trips()` on a 5-row synthetic DataFrame in
  a unit test and assert exactly which rows survive — no need to touch the
  real 9M-row dataset to verify logic.
- **Reusable.** Phase 3 (joins) and Phase 4 (aggregations) both need the
  *cleaned* data, not the raw data. A function is one clear seam between
  "raw" and "clean."
- **Honest about decisions.** Every filter is a business/data decision (drop?
  keep-but-flag? clip?). A named function with named parameters forces you to
  make — and document — that decision explicitly, rather than it being buried
  in a filter chain.

We are **not** yet writing this to a Bronze/Silver Delta table — that's Phase
6 (Delta Lake). Right now `clean_trips` just takes a DataFrame and returns a
DataFrame; it doesn't care where the input came from or where the output
goes. That's deliberate: the same function will get reused unchanged when we
wire it into a real bronze→silver pipeline later.

## Databricks note

In a Databricks Repo, this function would live in a `.py` file in the repo
(not the notebook) and get imported with `%run` or a package import, so it's
shared across notebooks and picked up by CI. We're deferring that move until
we have more than one function worth packaging (see the architecture note in
chat).


In [1]:
# Same read-and-cast setup from notebook 01 -- reused, not reinvented.
import os
from functools import reduce
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("DataForge-Phase2-Cleaning")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

RAW = "../data/raw"
assert os.path.exists(RAW), f"Can't find {RAW}. Current dir is: {os.getcwd()}"

TARGET_TYPES = {
    "VendorID": "long",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "RatecodeID": "double",
    "store_and_fwd_flag": "string",
    "PULocationID": "long",
    "DOLocationID": "long",
    "payment_type": "long",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
}

def read_and_cast(path: str) -> DataFrame:
    df = spark.read.parquet(path)
    df = df.toDF(*[c.lower() for c in df.columns])
    for col, target in TARGET_TYPES.items():
        df = df.withColumn(col, F.col(col.lower()).cast(target))
    return df.select(*TARGET_TYPES.keys())

paths = [
    f"{RAW}/yellow_tripdata_2023-01.parquet",
    f"{RAW}/yellow_tripdata_2023-02.parquet",
    f"{RAW}/yellow_tripdata_2023-03.parquet",
]
trips_raw = reduce(DataFrame.unionByName, [read_and_cast(p) for p in paths])

print("trips_raw ready:", trips_raw.count(), "rows")


trips_raw ready: 9384487 rows


## New API surface you'll need

A few PySpark tools you haven't used yet — look them up in the docs as you
work, don't just take my word for the signature:

- **`df.na.drop(subset=[...])`** — drop rows with nulls in specific columns.
  `df.na.fill({...})` does the opposite: replace nulls with a default per
  column. Deciding *drop vs. fill* per column is exactly the kind of decision
  this function should make explicit.
- **`df.dropDuplicates([...])`** — de-duplicate rows, optionally based on a
  subset of columns (e.g. two identical trips with the same pickup time,
  location, and fare are almost certainly the same trip logged twice).
- **`F.when(condition, value).otherwise(other_value)`** — the Spark
  equivalent of a `CASE WHEN`; you already used the null-counting version of
  this (`F.when(cond, 1)`) in notebook 01.

## Your task

Write `clean_trips(df: DataFrame) -> DataFrame` below. At minimum, make an
explicit, documented decision (a code comment is enough) for each of these,
using the issue counts you already measured in notebook 01 as your evidence:

1. **Null or zero `passenger_count`** — drop, or fill with a default?
2. **Negative `fare_amount` / `total_amount`** — these can't be real trips;
   drop them.
3. **Zero `trip_distance`** — could be a legitimate very-short/cancelled fare,
   or could be bad data. Decide, and justify it with a comment.
4. **`trip_distance > 100`** — pick a sane upper bound and drop trips beyond
   it (a NYC taxi trip over 100 miles is essentially always an error).
5. **`tpep_dropoff_datetime < tpep_pickup_datetime`** — drop.
6. **Pickup timestamps outside Jan–Mar 2023** — drop.
7. **Exact duplicate trips** — de-duplicate on a sensible column subset.

Don't worry about being "correct" — there often isn't one right answer here
(that's the point). Be able to justify each choice.

**Stretch goal:** have `clean_trips` also return a second DataFrame (or print
a summary) showing how many rows were removed by each rule, so the cleaning
step is auditable rather than a silent drop.


In [ ]:
def clean_trips(df: DataFrame) -> DataFrame:
    return (
        df
        # rule 2: negative fare/total can't be real trips -> keep only >= 0
        .filter((F.col("fare_amount") >= 0) & (F.col("total_amount") >= 0))
        # rule 5: dropoff before pickup is impossible -> keep only dropoff >= pickup
        .filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
        # rule 6: keep only pickups inside Jan–Mar 2023
        .filter(
            (F.col("tpep_pickup_datetime") >= F.lit("2023-01-01"))
            & (F.col("tpep_pickup_datetime") < F.lit("2023-04-01"))
        )
        # rule 4: keep only trip_distance <= 100 (miles)
        .filter(F.col("trip_distance") <= 100)
        # rule 3: zero-distance -> keep only if a positive fare exists (see note)
        .filter((F.col("trip_distance") > 0) | (F.col("fare_amount") > 0))
        # rule 1: null/zero passenger_count -> fill with 1 (see note)
        .withColumn(
            "passenger_count",
            F.when(
                F.col("passenger_count").isNull() | (F.col("passenger_count") == 0),
                F.lit(1),
            ).otherwise(F.col("passenger_count")),
        )
        # rule 7: de-duplicate on the fields that identify a distinct trip
        .dropDuplicates(
            [
                "tpep_pickup_datetime",
                "tpep_dropoff_datetime",
                "PULocationID",
                "DOLocationID",
                "trip_distance",
                "fare_amount",
            ]
        )
    )

In [3]:
trips_clean = clean_trips(trips_raw)
before, after = trips_raw.count(), trips_clean.count()
print(f"Before: {before:,}  After: {after:,}  Removed: {before - after:,} ({(before-after)/before:.2%})")


Before: 9,384,487  After: 9,301,798  Removed: 82,689 (0.88%)


In [4]:
trips_raw.filter((F.col("fare_amount") < 0) | (F.col("total_amount") < 0)).count()

79954

## Closing the open items

Two things left before we call Phase 2 done:

1. **Rule 3 evidence** — do zero-distance-but-paid trips correlate with a
   flat-fare `RatecodeID`? (NYC TLC's rate codes include things like
   `2` = JFK flat fare, `3` = Newark, `5` = negotiated fare — trips billed by
   agreement rather than the meter, which can legitimately show
   `trip_distance == 0` if the meter/GPS didn't log distance for a flat-rate
   ride.)
2. **Stretch goal** — a per-rule removal audit, so the cleaning step is
   auditable instead of a single opaque before/after count.


In [7]:
# Item 1: does zero-distance-but-paid correlate with a flat-fare RatecodeID?
(trips_raw
 .filter((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))
 .groupBy("RatecodeID")
 .count()
 .orderBy(F.desc("count"))
 .show())


+----------+-----+
|RatecodeID|count|
+----------+-----+
|       1.0|65858|
|       5.0|22772|
|      NULL|18780|
|       2.0| 8467|
|      99.0| 6475|
|       3.0| 1551|
|       4.0|   77|
|       6.0|   10|
+----------+-----+



## Stretch goal: per-rule removal audit

`clean_trips` applies all rules in one chain, so a single before/after count
can't tell you *which* rule did the work — you saw that reconciling it by
hand from notebook 01's numbers only got you approximately there, because
rules overlap (a row can fail two rules at once).

`clean_trips_audited` below applies the same rules one at a time, counting
the DataFrame after each step. This is deliberately **not** how you'd write
this in a real pipeline — it's much more expensive, because each `.count()`
is an action that forces Spark to rescan/recompute everything up to that
point (you flagged this exact anti-pattern yourself back in notebook 01).
We're paying that cost here on purpose, once, for visibility.

Notice the running total only ever *decreases or stays the same* — each rule
can only remove rows already surviving the previous ones, which is exactly
why order matters when rules overlap.


In [8]:
def clean_trips_audited(df: DataFrame) -> DataFrame:
    """Same rules as clean_trips, but prints the row count removed by each
    rule in isolation. Expensive (one .count() action per rule) -- for
    auditing/debugging only, never call this in a real pipeline."""
    remaining = df
    total_before = remaining.count()
    running = total_before
    print(f"{'start':40} {running:>12,}")

    steps = [
        ("rule 2: negative fare/total",
         lambda d: d.filter((F.col("fare_amount") >= 0) & (F.col("total_amount") >= 0))),
        ("rule 5: dropoff before pickup",
         lambda d: d.filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))),
        ("rule 6: pickup outside Jan-Mar 2023",
         lambda d: d.filter(
             (F.col("tpep_pickup_datetime") >= F.lit("2023-01-01")) &
             (F.col("tpep_pickup_datetime") < F.lit("2023-04-01"))
         )),
        ("rule 4: trip_distance > 100",
         lambda d: d.filter(F.col("trip_distance") <= 100)),
        ("rule 3: zero-distance & non-positive fare",
         lambda d: d.filter((F.col("trip_distance") > 0) | (F.col("fare_amount") > 0))),
        ("rule 7: duplicate trips",
         lambda d: d.dropDuplicates([
             "tpep_pickup_datetime", "tpep_dropoff_datetime",
             "PULocationID", "DOLocationID", "trip_distance", "fare_amount",
         ])),
    ]

    for label, step_fn in steps:
        remaining = step_fn(remaining)
        new_count = remaining.count()
        removed = running - new_count
        print(f"{label:40} {new_count:>12,}   (-{removed:,})")
        running = new_count

    # rule 1 is a fill, not a filter -- it never removes rows, so it's applied
    # last and doesn't appear in the audit trail above.
    remaining = remaining.withColumn(
        "passenger_count",
        F.when(
            F.col("passenger_count").isNull() | (F.col("passenger_count") == 0),
            F.lit(1),
        ).otherwise(F.col("passenger_count")),
    )

    print(f"{'total removed':40} {total_before - running:>12,}   ({(total_before - running) / total_before:.2%})")
    return remaining


trips_clean_audited = clean_trips_audited(trips_raw)


start                                       9,384,487
rule 2: negative fare/total                 9,304,533   (-79,954)
rule 5: dropoff before pickup               9,303,897   (-636)
rule 6: pickup outside Jan-Mar 2023         9,303,758   (-139)
rule 4: trip_distance > 100                 9,303,469   (-289)
rule 3: zero-distance & non-positive fare    9,301,799   (-1,670)
rule 7: duplicate trips                     9,301,798   (-1)
total removed                                  82,689   (0.88%)


## Further reading

- [PySpark API Reference — `DataFrameNaFunctions`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameNaFunctions.html) — the full `na.drop()` / `na.fill()` API used for rule 1.
- [PySpark API Reference — `functions.when`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html) — the `CASE WHEN` equivalent used throughout `clean_trips`.
- [PySpark API Reference — `dropDuplicates`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html) — used for rule 7; note the docs' warning about non-deterministic row selection when duplicates aren't byte-identical.
- [NYC TLC Trip Record Data — data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) — official field definitions, including what each `RatecodeID` value means (used for the rule 3 investigation).

> **Version note:** these links point at the latest PySpark docs (4.2.0); our
> Docker image runs Spark 3.5.3 — switch the docs' version selector to 3.5.x
> if something looks inconsistent.
